# Grounded Research Adjudication — Review Journey

**Journey purpose:** Inspect the end-to-end adjudication pipeline as it evolves from planning artifacts into live implementation.

**Notebook mode:** review surface for `planned | partial | stub | fixture | live` phases.

**Related docs:**
- [`CLAUDE.md`](../../CLAUDE.md)
- [`docs/PLAN.md`](../PLAN.md)
- [`docs/DOMAIN_MODEL.md`](../DOMAIN_MODEL.md)
- [`docs/CONTRACTS.md`](../CONTRACTS.md)
- [`docs/ARCHITECTURE_ONE_PAGE.md`](../ARCHITECTURE_ONE_PAGE.md)
- [`docs/adr/0001-adjudication-first-scope.md`](../adr/0001-adjudication-first-scope.md)
- [`docs/adr/0002-approved-external-reuse-strategy.md`](../adr/0002-approved-external-reuse-strategy.md)

**Current review artifacts:**
- models: `src/grounded_research/models.py`
- fixture: `tests/fixtures/session_storage_bundle.json`
- contracts: `docs/CONTRACTS.md`


In [ ]:
"""Setup: imports, root discovery, and review helpers."""
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_project_root(start: Path) -> Path:
    """Walk upward until the repo root containing CLAUDE.md is found."""
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "CLAUDE.md").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("Could not locate project root from current working directory")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.grounded_research.models import (
    AnalystRun,
    ArbitrationResult,
    Claim,
    ClaimLedger,
    Counterargument,
    Dispute,
    DownstreamHandoff,
    EvidenceBundle,
    EvidenceItem,
    FinalReport,
    PhaseTrace,
    PipelineState,
    RawClaim,
    Recommendation,
    ResearchQuestion,
    SourceRecord,
    VerificationQueryBatch,
    DISPUTE_ROUTING,
)

FIXTURE_PATH = PROJECT_ROOT / "tests" / "fixtures" / "session_storage_bundle.json"


def show(title: str, payload: object) -> None:
    print(f"\n=== {title} ===")
    pprint(payload)


print(f"Project root: {PROJECT_ROOT}")
print(f"Fixture exists: {FIXTURE_PATH.exists()}")


## Phase -1: Thesis Falsification

- **Purpose:** Validate that multi-model disagreement produces decision-relevant signal before building the full adjudication system.
- **Input → Output:** `question: str` + imported evidence bundle → 3 `AnalystRun`-shaped outputs (or equivalent analyst artifacts) + minimal claim extraction artifact + manual disagreement review
- **Acceptance criteria:** Disagreements are not mostly framing noise; at least some are decision-relevant; fresh evidence plausibly sharpens answers.
- **Status:** `planned`
- **Execution mode:** `planned`
- **Promotion path:** `planned` → `live`
- **Current gap:** real `llm_client` calls, minimal extracted-claim review artifacts, and baseline comparison paths are not wired yet.


In [ ]:
phase_m1 = {
    "phase": -1,
    "name": "thesis_falsification",
    "status": "planned",
    "execution_mode": "planned",
    "input": {
        "question": "Should a team use Redis or PostgreSQL for session storage?",
        "evidence_bundle": str(FIXTURE_PATH),
        "baselines": ["manual/research_v3", "STORM or GPT Researcher when available"],
    },
    "expected_output": {
        "analyst_count": 3,
        "minimal_claim_artifact": "compact list of extracted claims or claim stubs per analyst",
        "manual_review_focus": [
            "Are disagreements decision-relevant?",
            "Would fresh evidence materially change the answer?",
            "Do analysts surface genuinely different tradeoffs?",
        ],
    },
}
show("Phase -1 Artifact", phase_m1)


## Phase 0: Domain Model, Contracts, Trace, and Review Surface

- **Purpose:** Finish design-method steps 3-5 before runtime implementation.
- **Input → Output:** planning docs + draft models → canonical domain model, contracts, and trace skeleton
- **Acceptance criteria:** field-level entities exist; inter-phase contracts are explicit; `PipelineState` serializes; notebook remains runnable.
- **Status:** `partial`
- **Execution mode:** `stub`
- **Promotion path:** `partial` → `live`
- **Current gap:** an adopted `pyproject.toml`, `config/config.yaml`, `prompts/`, and a dry-run CLI are not yet in a verified Phase 0 milestone.


In [ ]:
state = PipelineState()
state.phase_traces.append(
    PhaseTrace(
        phase="init",
        started_at=state.started_at,
        succeeded=True,
        output_summary="Initialized empty PipelineState skeleton.",
    )
)
state_json = state.model_dump_json(indent=2)
state_rt = PipelineState.model_validate_json(state_json)
assert state_rt.run_id == state.run_id
assert state_rt.current_phase == "init"

print(f"Run ID: {state.run_id}")
print(f"Trace skeleton size: {len(state_json)} bytes")
print("Round-trip serialization: OK")
print("Current gap: an adopted pyproject.toml, config/config.yaml, prompts/, and CLI scaffold are not in a verified Phase 0 milestone yet.")


## Phase 1: Ingest

- **Purpose:** Normalize imported evidence into internal schemas without losing provenance.
- **Input → Output:** upstream bundle JSON/YAML → `EvidenceBundle`
- **Acceptance criteria:** every `EvidenceItem.source_id` resolves; timestamps survive; known evidence gaps are explicit.
- **Status:** `partial`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** adapters for `research_v3`, STORM, and GPT Researcher are not implemented yet.


In [ ]:
raw = json.loads(FIXTURE_PATH.read_text())
question = ResearchQuestion(**raw["question"])
sources = [SourceRecord(**s) for s in raw["sources"]]
evidence = [EvidenceItem(**e) for e in raw["evidence"]]

bundle = EvidenceBundle(
    question=question,
    sources=sources,
    evidence=evidence,
    gaps=raw["gaps"],
    imported_from=raw["imported_from"],
)

source_ids = {s.id for s in bundle.sources}
orphans = [e.id for e in bundle.evidence if e.source_id not in source_ids]
assert not orphans, f"Orphan evidence items: {orphans}"
assert bundle.imported_from == "manual"

show("Phase 1 Bundle Summary", {
    "question": bundle.question.text,
    "source_count": len(bundle.sources),
    "evidence_count": len(bundle.evidence),
    "gap_count": len(bundle.gaps),
})


## Phase 2a: Single Analyst

- **Purpose:** Prove one analyst can produce valid structured output before scaling to three.
- **Input → Output:** `ResearchQuestion + EvidenceBundle` → `AnalystRun`
- **Acceptance criteria:** valid `AnalystRun`; evidence-linked claims; at least one claim, assumption, recommendation, and counterargument.
- **Status:** `stub`
- **Execution mode:** `stub`
- **Promotion path:** `stub` → `live`
- **Current gap:** `prompts/analyst.yaml` and live `llm_client` structured calls are not wired.


In [ ]:
analyst_alpha = AnalystRun(
    analyst_label="Alpha",
    frame="verification_first",
    model="stub/not-a-real-model",
    claims=[
        RawClaim(
            id="RC-stub0001",
            statement="Redis provides significantly lower read latency at the target concurrency.",
            evidence_ids=["E-fix00001", "E-fix00003"],
            confidence="high",
            reasoning="Official Redis performance characteristics plus the direct benchmark support the latency advantage.",
        ),
        RawClaim(
            id="RC-stub0002",
            statement="Redis should be the default recommendation for this workload when performance is prioritized.",
            evidence_ids=["E-fix00003", "E-fix00004"],
            confidence="medium",
            reasoning="Redis has lower latency and more headroom on write throughput than PostgreSQL in the benchmark.",
        ),
    ],
    assumptions=[
        {
            "statement": "The team can tolerate introducing a new operational dependency.",
            "basis": "A managed Redis service is available in the target environment.",
            "challenged": False,
        }
    ],
    recommendations=[
        Recommendation(
            statement="Use Redis as the session store if the primary objective is minimizing session latency.",
            supporting_claim_ids=["RC-stub0001", "RC-stub0002"],
            conditions="This holds if added infrastructure complexity is acceptable.",
        )
    ],
    counterarguments=[
        Counterargument(
            target="Redis recommendation",
            argument="Redis adds new infrastructure cost and may accept session loss during cluster incidents.",
            evidence_ids=["E-fix00005", "E-fix00006"],
        )
    ],
    summary="Alpha favors Redis for raw performance but acknowledges infrastructure and reliability tradeoffs.",
)

assert analyst_alpha.succeeded
assert analyst_alpha.claims
assert analyst_alpha.assumptions
assert analyst_alpha.recommendations
assert analyst_alpha.counterarguments
show("Phase 2a Analyst", analyst_alpha.model_dump())


## Phase 2b: Three Independent Analysts

- **Purpose:** Produce useful divergence over the same evidence set using three blind analysts.
- **Input → Output:** `ResearchQuestion + EvidenceBundle` → `list[AnalystRun]`
- **Acceptance criteria:** analysts use different frames; no analyst output appears in another analyst input; at least some useful divergence appears.
- **Status:** `stub`
- **Execution mode:** `stub`
- **Promotion path:** `stub` → `live`
- **Current gap:** parallel live calls and min-success abort logic are not wired.


In [ ]:
analyst_beta = AnalystRun(
    analyst_label="Beta",
    frame="structured_decomposition",
    model="stub/not-a-real-model",
    claims=[
        RawClaim(
            id="RC-stub0003",
            statement="PostgreSQL with tuning can support 50K concurrent sessions with acceptable latency.",
            evidence_ids=["E-fix00008", "E-fix00009"],
            confidence="high",
            reasoning="The case study shows sustained production reliability with tuned connection pooling and maintenance.",
        ),
        RawClaim(
            id="RC-stub0004",
            statement="For teams with existing PostgreSQL expertise, PostgreSQL-backed sessions may have lower total cost of ownership.",
            evidence_ids=["E-fix00005", "E-fix00010"],
            confidence="medium",
            reasoning="The cost comparison and academic study both emphasize operational simplicity and reuse of existing expertise.",
        ),
    ],
    assumptions=[
        {
            "statement": "The team is willing to tune pooling and vacuuming for session tables.",
            "basis": "They already operate PostgreSQL and can absorb moderate tuning work.",
            "challenged": False,
        }
    ],
    recommendations=[
        Recommendation(
            statement="Prefer PostgreSQL session storage when operational simplicity and existing team expertise outweigh Redis latency gains.",
            supporting_claim_ids=["RC-stub0003", "RC-stub0004"],
            conditions="This holds if p99 under 5ms is acceptable for the application.",
        )
    ],
    counterarguments=[
        Counterargument(
            target="PostgreSQL recommendation",
            argument="Redis still provides materially better latency and throughput headroom.",
            evidence_ids=["E-fix00001", "E-fix00003", "E-fix00004"],
        )
    ],
    summary="Beta favors PostgreSQL for total cost of ownership and reuse of existing operational expertise.",
)

analyst_gamma = AnalystRun(
    analyst_label="Gamma",
    frame="step_back_abstraction",
    model="stub/not-a-real-model",
    claims=[
        RawClaim(
            id="RC-stub0005",
            statement="Both Redis and PostgreSQL satisfy the throughput requirement for 50K concurrent users.",
            evidence_ids=["E-fix00003", "E-fix00004"],
            confidence="high",
            reasoning="The benchmark shows both systems exceed the write throughput implied by the concurrency target.",
        ),
        RawClaim(
            id="RC-stub0006",
            statement="The decision should hinge more on failure mode tolerance and team expertise than on raw throughput.",
            evidence_ids=["E-fix00006", "E-fix00007", "E-fix00010"],
            confidence="medium",
            reasoning="The evidence suggests the operational and reliability tradeoffs dominate once both systems meet the base load requirement.",
        ),
    ],
    assumptions=[
        {
            "statement": "Application requirements tolerate p99 latency in the low single-digit milliseconds.",
            "basis": "No evidence in the bundle says sub-millisecond latency is mandatory.",
            "challenged": False,
        }
    ],
    recommendations=[
        Recommendation(
            statement="Choose the system whose failure modes and operational model the team is more willing to own; raw throughput is not the deciding factor here.",
            supporting_claim_ids=["RC-stub0005", "RC-stub0006"],
            conditions="This holds if the application does not require Redis-level latency specifically.",
        )
    ],
    counterarguments=[
        Counterargument(
            target="failure-mode-first recommendation",
            argument="If latency is a product differentiator, Redis performance may still justify its operational cost.",
            evidence_ids=["E-fix00001", "E-fix00003"],
        )
    ],
    summary="Gamma reframes the choice as failure-mode tolerance and operational ownership rather than raw performance alone.",
)

analyst_runs = [analyst_alpha, analyst_beta, analyst_gamma]
assert len({a.analyst_label for a in analyst_runs}) == 3
assert len({a.frame for a in analyst_runs}) == 3
show("Phase 2b Analysts", [a.model_dump() for a in analyst_runs])


## Phase 3a: Claim Extraction

- **Purpose:** Flatten analyst outputs into reviewable raw claims with provenance.
- **Input → Output:** `list[AnalystRun]` → `list[RawClaim]`
- **Acceptance criteria:** every `RawClaim` traces to an analyst; no claim text is invented.
- **Status:** `partial`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** provenance is still carried in a side map rather than a dedicated normalized extraction artifact.


In [ ]:
raw_claims = []
raw_claim_provenance = {}
for analyst in analyst_runs:
    for claim in analyst.claims:
        raw_claims.append(claim)
        raw_claim_provenance[claim.id] = analyst.analyst_label

assert raw_claims
assert all(rc.id in raw_claim_provenance for rc in raw_claims)
show("Phase 3a Raw Claims", {
    "count": len(raw_claims),
    "by_analyst": raw_claim_provenance,
})


## Phase 3b: Semantic Deduplication

- **Purpose:** Merge semantically equivalent raw claims into canonical claims while preserving provenance.
- **Input → Output:** `list[RawClaim]` → `list[Claim]`
- **Acceptance criteria:** canonical claims preserve raw-claim IDs, analyst sources, and evidence links; distinct claims remain separate.
- **Status:** `stub`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** live semantic grouping is not wired; this notebook uses manually defined equivalence classes.


In [ ]:
canonical_claims = [
    Claim(
        id="C-000001",
        statement="Redis provides materially lower session-read latency at the target concurrency.",
        source_raw_claim_ids=["RC-stub0001"],
        analyst_sources=["Alpha"],
        evidence_ids=["E-fix00001", "E-fix00003"],
        confidence="high",
    ),
    Claim(
        id="C-000002",
        statement="PostgreSQL can support 50K concurrent sessions with acceptable latency after tuning.",
        source_raw_claim_ids=["RC-stub0003"],
        analyst_sources=["Beta"],
        evidence_ids=["E-fix00008", "E-fix00009"],
        confidence="high",
    ),
    Claim(
        id="C-000003",
        statement="Both Redis and PostgreSQL satisfy the throughput requirement for this workload.",
        source_raw_claim_ids=["RC-stub0005"],
        analyst_sources=["Gamma"],
        evidence_ids=["E-fix00003", "E-fix00004"],
        confidence="high",
    ),
    Claim(
        id="C-000004",
        statement="For teams with existing PostgreSQL expertise, PostgreSQL-backed sessions may have lower total cost of ownership.",
        source_raw_claim_ids=["RC-stub0004", "RC-stub0006"],
        analyst_sources=["Beta", "Gamma"],
        evidence_ids=["E-fix00005", "E-fix00006", "E-fix00007", "E-fix00010"],
        confidence="medium",
    ),
    Claim(
        id="C-000005",
        statement="Redis should be the default recommendation when latency is the top priority.",
        source_raw_claim_ids=["RC-stub0002"],
        analyst_sources=["Alpha"],
        evidence_ids=["E-fix00003", "E-fix00004"],
        confidence="medium",
    ),
]

assert all(c.source_raw_claim_ids for c in canonical_claims)
assert all(c.analyst_sources for c in canonical_claims)
show("Phase 3b Canonical Claims", [c.model_dump() for c in canonical_claims])


## Phase 3c: Claim Ledger and Disputes

- **Purpose:** Build the canonical product artifact and localize conflicts.
- **Input → Output:** `list[Claim]` → `ClaimLedger`
- **Acceptance criteria:** disputes reference real claim IDs; routes match the deterministic routing table; humans can inspect the conflicts.
- **Status:** `stub`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** dispute classification is manually specified here; live classification prompt is not wired.


In [ ]:
disputes = [
    Dispute(
        id="D-000001",
        dispute_type="interpretive_conflict",
        route=DISPUTE_ROUTING["interpretive_conflict"],
        claim_ids=["C-000004", "C-000005"],
        description="Whether lower operational complexity and TCO should outweigh Redis latency gains for this team.",
        severity="decision_critical",
        resolved=False,
    )
]

ledger = ClaimLedger(claims=canonical_claims, disputes=disputes)
assert ledger.disputes[0].route == "arbitrate"
show("Phase 3c Claim Ledger", ledger.model_dump())


## Phase 4a: Verification Query Generation

- **Purpose:** Generate targeted searches for decision-critical disputes.
- **Input → Output:** verify-worthy `list[Dispute]` → `list[VerificationQueryBatch]`
- **Acceptance criteria:** each query batch maps to a real dispute and is specific enough to plausibly add new evidence.
- **Status:** `stub`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** live counterfactual search query generation is not wired.


In [ ]:
verification_queries = [
    VerificationQueryBatch(
        dispute_id="D-000001",
        queries=[
            "session storage operational incidents redis vs postgresql 2025 2026",
            "postgresql session storage total cost ownership existing team expertise",
            "redis session storage latency business impact versus operational complexity",
        ],
        rationale="Clarify whether the operational/reliability tradeoff outweighs Redis latency gains for teams already operating PostgreSQL.",
    )
]

assert verification_queries[0].dispute_id == disputes[0].id
show("Phase 4a Verification Queries", [q.model_dump() for q in verification_queries])


## Phase 4b: Arbitration and Ledger Update

- **Purpose:** Use new evidence to update claim status and resolve a dispute.
- **Input → Output:** `ClaimLedger + list[VerificationQueryBatch] + fresh evidence` → updated `ClaimLedger + list[ArbitrationResult]`
- **Acceptance criteria:** arbitration cites new evidence; claim updates match the verdict; resolved disputes are marked resolved.
- **Status:** `stub`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** fresh search retrieval and live arbitration are not wired.


In [ ]:
verification_source = SourceRecord(
    id="S-ver00001",
    url="https://ops.example.com/session-storage-postmortem-2026",
    title="2026 Postmortem: Session Store Migration and Reliability Outcomes",
    source_type="web_search",
    quality_tier="reliable",
    published_at="2026-02-10T00:00:00Z",
    retrieved_at="2026-03-21T12:00:00Z",
    recency_score=0.88,
)
verification_evidence = EvidenceItem(
    id="E-ver00001",
    source_id="S-ver00001",
    content="Teams with existing PostgreSQL expertise reported lower total incident volume after avoiding a new Redis dependency, while session latency remained under 5ms after tuning.",
    content_type="summary",
    relevance_note="Fresh evidence bearing directly on the TCO-versus-latency dispute.",
    extraction_method="manual",
)

bundle.sources.append(verification_source)
bundle.evidence.append(verification_evidence)

arbitration_results = [
    ArbitrationResult(
        dispute_id="D-000001",
        verdict="supported",
        new_evidence_ids=["E-ver00001"],
        reasoning="The new evidence directly supports the operational/TCO interpretation for teams already invested in PostgreSQL while not refuting Redis' raw performance advantage.",
        claim_updates={
            "C-000004": "supported",
            "C-000005": "inconclusive",
        },
    )
]

for claim in ledger.claims:
    if claim.id in arbitration_results[0].claim_updates:
        claim.status = arbitration_results[0].claim_updates[claim.id]
        claim.status_reason = arbitration_results[0].reasoning

ledger.arbitration_results.extend(arbitration_results)
ledger.disputes[0].resolved = True
ledger.disputes[0].resolution_summary = "Fresh evidence favored the TCO/operational interpretation for this team profile."

show("Phase 4b Updated Ledger", ledger.model_dump())


## Phase 5: Grounded Export and Downstream Handoff

- **Purpose:** Render the adjudicated state for both human review and downstream systems.
- **Input → Output:** `PipelineState` → `FinalReport + report.md + trace.json + DownstreamHandoff`
- **Acceptance criteria:** recommendation cites real claims; cited claims resolve to evidence; unresolved disputes remain visible when present.
- **Status:** `stub`
- **Execution mode:** `fixture`
- **Promotion path:** `fixture` → `live`
- **Current gap:** live synthesis prompt and file export are not wired.


In [ ]:
report = FinalReport(
    title="Session Storage Recommendation Review",
    question=bundle.question.text,
    recommendation="Recommend PostgreSQL-backed session storage for this team profile, supported by C-000004 and bounded by the performance considerations in C-000001 and C-000005.",
    alternatives=[
        "Use Redis if sub-millisecond latency is a hard product requirement (see C-000001)."
    ],
    disagreement_summary="The central dispute was whether Redis latency gains outweighed PostgreSQL operational simplicity and TCO. Arbitration favored the TCO interpretation for this team profile.",
    evidence_gaps=bundle.gaps,
    flip_conditions=[
        "If the application requires sub-millisecond session reads as a product requirement.",
        "If the team is already operating Redis for adjacent workloads."
    ],
    cited_claim_ids=["C-000004", "C-000001", "C-000005"],
)

handoff = DownstreamHandoff(
    question=bundle.question,
    claim_ledger=ledger,
    sources=bundle.sources,
    evidence=bundle.evidence,
)

state.question = bundle.question
state.evidence_bundle = bundle
state.analyst_runs = analyst_runs
state.claim_ledger = ledger
state.report = report
state.current_phase = "export"
state.phase_traces.extend([
    PhaseTrace(phase="ingest", started_at=state.started_at, succeeded=True, output_summary="Fixture bundle loaded into EvidenceBundle."),
    PhaseTrace(phase="analyze", started_at=state.started_at, succeeded=True, output_summary="Three stub analysts created."),
    PhaseTrace(phase="canonicalize", started_at=state.started_at, succeeded=True, output_summary="Claims deduplicated and dispute localized."),
    PhaseTrace(phase="adjudicate", started_at=state.started_at, succeeded=True, output_summary="One dispute arbitrated with fresh evidence."),
    PhaseTrace(phase="export", started_at=state.started_at, succeeded=True, output_summary="FinalReport and DownstreamHandoff assembled."),
])

ledger_claim_ids = {c.id for c in ledger.claims}
assert set(report.cited_claim_ids).issubset(ledger_claim_ids)
for claim_id in report.cited_claim_ids:
    claim = ledger.claim_by_id(claim_id)
    assert claim is not None
    assert claim.evidence_ids
    for evidence_id in claim.evidence_ids:
        assert any(e.id == evidence_id for e in bundle.evidence)

show("Phase 5 Report", report.model_dump())
show("Phase 5 Handoff Summary", {
    "target": handoff.downstream_target,
    "claim_count": len(handoff.claim_ledger.claims),
    "source_count": len(handoff.sources),
    "evidence_count": len(handoff.evidence),
})
print(f"Serialized trace bytes: {len(state.model_dump_json())}")
